# Fase 4c — Generación del dataset de preguntas (TEST / Golden Dataset)

Genera el conjunto de evaluación  a partir de los macro-contextos de `dataset_test.json` (Fase 3a). A diferencia del dataset de TRAIN (Fase 4a), este se usa para evaluar el sistema RAG completo (recuperación + generación), por lo que es más exigente: cada pregunta debe ser autosuficiente (sin referencias ambiguas como "este tratamiento"), las preguntas "Compleja" deben exigir un verdadero *multi-salto* (combinar dos fragmentos distintos), y se añade un cuarto tipo, **"Negativa"** (pregunta plausible pero sin respuesta en el texto, para comprobar que el sistema reconoce la ausencia de información). Cada pregunta incluye además una `respuesta_ideal`.

1. **Generación con Gemini** (`generar_preguntas_gemini`, `procesar_test`): 4 preguntas por macro-contexto (Directa, Caso, Compleja, Negativa), con su cita literal (excepto la Negativa) acotada a fragmentos de máximo 500 caracteres, ya que se compararán individualmente contra los chunks recuperados.
2. **Doble auditoría** (`auditar_dataset`):
   - **Fidelidad**: descarta preguntas cuya cita no se localiza de forma suficientemente fiel (fuzzy matching ≥ 90%) en el macro-contexto original, o cuyos fragmentos superan los 500 caracteres.
   - **Supervivencia al chunking**: para cada una de las cuatro estrategias de la Fase 3b, comprueba si la cita validada sigue siendo recuperable dentro de un único chunk o de un par de chunks adyacentes del mismo documento, informando la tasa de recuperabilidad máxima por estrategia y cuántas citas quedan "destruidas" (repartidas de forma no recuperable entre más de dos chunks).
3. **Auditoría de granularidad** (`auditar_granularidad_preguntas`) del dataset de test final (`preguntas_evaluacion_test_final.json`): distribución por tipo, longitudes y control de integridad de campos.

> **Nota de seguridad:** la clave de la API de Gemini se ha sustituido por un marcador de posición (`TU_GOOGLE_API_KEY_AQUI`); debe configurarse como variable de entorno o sustituirse por una clave propia antes de ejecutar el notebook.

In [1]:
pip install thefuzz python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 89.8 MB/s eta 0:00:00


In [2]:
pip install --upgrade google-generativeai

In [ ]:
import json
import random
import time
import google.generativeai as genai

# ==========================================
# CONFIGURACIÓN DE GEMINI
# ==========================================
# Pon tu API Key aquí. (En producción es mejor usar variables de entorno)
API_KEY = "TU_GOOGLE_API_KEY_AQUI"
genai.configure(api_key=API_KEY)

# Usamos el modelo más potente y reciente que has especificado
MODELO_ELEGIDO = 'gemini-2.5-pro'
model = genai.GenerativeModel(MODELO_ELEGIDO)

# ==========================================
# LA RULETA DE ROLES
# ==========================================
ROLES = [
    "un médico residente de primer año con dudas rápidas",
    "un especialista adjunto repasando un protocolo específico",
    "un enfermero de urgencias buscando una dosis exacta",
    "un médico de familia en consulta con un paciente",
    "un tribunal de examen médico formulando una pregunta tipo test"
]

INSTRUCCIONES_SISTEMA = """
    Eres un experto en la extracción de datos clínicos y evaluación de sistemas de IA (RAG),
    creando datasets de test.

    REGLAS ESTRICTAS DE GENERACIÓN:

    1. ANTI-BUCLE: Sé creativo con la formulación. PROHIBIDO empezar todas las preguntas igual.

    2. AUTOSUFICIENCIA DE LA PREGUNTA: Imagina que la pregunta se lee en un post-it perdido.
      NUNCA uses términos ambiguos como "este tratamiento", "dicha enfermedad" o
      "el paciente del texto". Nombra explícitamente el fármaco, enfermedad o contexto
      en la propia pregunta.

    3. VARIEDAD LÉXICA: PROHIBIDO usar exactamente las mismas palabras clave del texto en
      la pregunta. Obliga al sistema a pensar. Usa sinónimos, parafraseo, lenguaje clínico
      alternativo o descripciones de síntomas en lugar del nombre técnico.

    4. AUTOSUFICIENCIA DE LA CITA (CRÍTICO):
      La cita debe contener TANTO el sujeto clínico (medicamento, patología, tipo de
      paciente) COMO la información que responde a la pregunta.
      Alguien que lea SOLO la cita, sin ningún contexto externo, debe poder responder
      la pregunta de forma correcta y completa.

      MAL EJEMPLO:
      Pregunta: "¿Cuál es la pauta de administración de Beta-micoter en adultos?"
      Cita: "Adultos: aplicar una fina capa 2 veces al día."
      → INVÁLIDO: la cita no menciona Beta-micoter. No se sabe a qué medicamento
        corresponde esa pauta.

      BUEN EJEMPLO:
      Pregunta: "¿Cuál es la pauta de administración de Beta-micoter en adultos?"
      Cita: "Beta-micoter crema. Adultos: aplicar una fina capa 2 veces al día."
      → VÁLIDO: la cita ancla el medicamento y la pauta en el mismo fragmento.

      REGLA PRÁCTICA: Antes de escribir la cita, hazte esta pregunta:
      "Si alguien lee SOLO esta cita, ¿puede identificar sin ambigüedad el sujeto
      clínico al que aplica la información?" Si la respuesta es NO, amplía la cita
      hacia arriba en el texto hasta incluir la cabecera o contexto necesario.

    5. REGLA ESPECIAL PARA PREGUNTAS COMPLEJAS (multi-salto):
      Una pregunta Compleja REAL exige que el sistema RAG encuentre e integre DOS
      fragmentos de información distintos del texto para construir la respuesta.

      DEFINICIÓN DE MULTI-SALTO: la pregunta NO proporciona ninguno de los dos datos
      en su enunciado. Ambos deben ser recuperados por el sistema.

      MAL EJEMPLO (falso multi-salto):
      Pregunta: "Sabiendo que los corticoides causan adelgazamiento de la piel,
                  ¿qué tipo de apósito se debe evitar para no aumentar su absorción?"
      → INVÁLIDO: el enunciado ya da la premisa ("adelgazamiento de la piel").
        El RAG solo necesita encontrar un fragmento. No hay salto real.

      BUEN EJEMPLO (multi-salto real):
      Pregunta: "¿Qué práctica de vendaje está contraindicada con los corticoides
                  tópicos y qué consecuencia sistémica específica justifica evitarla?"
      Cita: "Evitar vendajes oclusivos pues podrían favorecer la absorción sistémica.
              Las reacciones adversas con el uso de corticosteroides tópicos son mayores
              si se utilizan vendajes oclusivos."
      → VÁLIDO: el RAG debe recuperar (1) qué vendaje está contraindicado y
        (2) por qué, integrando ambos fragmentos para responder completamente.

      REGLA PRÁCTICA: Antes de escribir la pregunta Compleja, hazte esta prueba:
      "¿Podría responderse esta pregunta recuperando UN SOLO fragmento del texto?"
      Si la respuesta es SÍ, la pregunta NO es multi-salto. Redísela para que
      requiera obligatoriamente dos fragmentos distintos.

    TAREA: Genera un JSON estricto con 4 preguntas:
    1. "Directa": Factual y sencilla.
    2. "Caso": Aplicación a un paciente ficticio.
    3. "Compleja": Requiere recuperar e integrar DOS fragmentos distintos del texto
      (multi-salto real según las reglas anteriores).
    4. "Negativa": Pregunta médicamente plausible y relacionada con la temática,
      pero cuya respuesta NO está en el texto.

    REGLA CITA LITERAL:
    Para las preguntas 1 (Directa), 2 (Caso) y 3 (Compleja), extrae la
    "cita_literal" EXACTA del texto que justifica la respuesta.

    Cada fragmento de la cita no debe superar los 500 caracteres, ya que
    será comparado individualmente contra los chunks recuperados.

    Para preguntas Directa y Caso: la cita contiene un único fragmento
    mínimo que incluya el sujeto clínico y la información que responde
    la pregunta.

    Para la pregunta Compleja: la cita contiene DOS fragmentos separados
    por [...], cada uno respetando el límite de 500 caracteres
    individualmente. Ambos fragmentos deben pertenecer a la misma
    sección del documento con un snetido lógico.

    Copia y pega EXACTAMENTE sin cambiar ni una coma.
    Devuelve ÚNICAMENTE este JSON (sin formato Markdown extra, solo el array JSON):
    [
      {
        "tipo": "Directa",
        "pregunta": "...",
        "cita_literal": "...",
        "respuesta_ideal": "..."
      },
      {
        "tipo": "Caso",
        "pregunta": "...",
        "cita_literal": "...",
        "respuesta_ideal": "..."
      },
      {
        "tipo": "Compleja",
        "pregunta": "...",
        "cita_literal": "...",
        "respuesta_ideal": "..."
      },
      {
        "tipo": "Negativa",
        "pregunta": "...",
        "cita_literal": "NO APLICA",
        "respuesta_ideal": "El documento no proporciona información sobre..."
      }
    ]
"""

# Inicializamos el modelo con las instrucciones del sistema
model = genai.GenerativeModel(
    model_name=MODELO_ELEGIDO,
    system_instruction=INSTRUCCIONES_SISTEMA
)

def generar_preguntas_gemini(macro_contexto_texto, reintentos=3):
    """Llama a Gemini con el prompt dinámico, fuerza salida JSON y maneja reintentos."""

    rol = random.choice(ROLES)

    prompt = f"Asumiendo el rol de: {rol}, genera las 4 preguntas para el siguiente texto:\n\n{macro_contexto_texto}"

    configuracion = genai.GenerationConfig(
        response_mime_type="application/json",
        temperature=0.4
    )

    for intento in range(reintentos):
        try:
            respuesta = model.generate_content(prompt, generation_config=configuracion)

            if not respuesta.parts:
                 print(f"  [!] Respuesta bloqueada por filtros de seguridad de la API (Intento {intento+1}).")
                 time.sleep(2)
                 continue

            texto_limpio = respuesta.text.strip()

            if texto_limpio.startswith("```json"):
                texto_limpio = texto_limpio[7:-3]
            elif texto_limpio.startswith("```"):
                texto_limpio = texto_limpio[3:-3]

            return json.loads(texto_limpio.strip())

        except json.JSONDecodeError:
            print(f"  [!] Fallo al parsear JSON (Intento {intento+1}). Reintentando...")
            time.sleep(2)
        except Exception as e:
            if "429" in str(e):
                print(f"  [!] Límite de API (429). Esperando 10s... (Intento {intento+1})")
                time.sleep(10)
            else:
                print(f"  [!] Error de API: {e} (Intento {intento+1})")
                time.sleep(2)

    raise Exception("Se agotaron los reintentos para este bloque.")

def procesar_test(ruta_test='dataset_test.json', ruta_salida='preguntas_evaluacion_test.json'):
    """
    Recorre todos los macro-contextos de test, genera con
    generar_preguntas_gemini sus 4 preguntas (incluida la 'Negativa', sin
    respuesta en el texto), añade los metadatos de origen y guarda el
    resultado conjunto en un JSON, avisando si alguna cita supera los 500
    caracteres.
    """
    print(f"INICIANDO GENERACIÓN DE PREGUNTAS (TEST) CON GEMINI...")

    with open(ruta_test, 'r', encoding='utf-8') as f:
        datos_test = json.load(f)

    resultados_finales = []
    total = len(datos_test)

    for i, bloque in enumerate(datos_test, 1):
        print(f"[{i}/{total}] Procesando bloque de: {bloque.get('source', 'Desconocido')[:30]}...")

        try:
            # Llamamos a Gemini
            preguntas_generadas = generar_preguntas_gemini(bloque["text"])

            # Procesamos y validamos cada pregunta
            for p in preguntas_generadas:
                cita = p.get("cita_literal", "")

                # Auditoría de longitud en tiempo real
                if cita not in ["NO APLICA", "NULL"] and len(cita) > 500:
                    print(f"    [Aviso] Cita un poco larga detectada ({len(cita)} chars) en tipo '{p['tipo']}'")

                # Inyectamos metadatos
                p["macro_id_origen"] = bloque["macro_id"]
                p["source"] = bloque.get("source", "Desconocido")
                resultados_finales.append(p)

            # Pausa para no saturar los límites de la API
            time.sleep(2)

        except Exception as e:
            print(f" [ERROR CRÍTICO] Saltando bloque {i}: {e}")
            continue

    # Guardamos el resultado final
    with open(ruta_salida, 'w', encoding='utf-8') as f:
        json.dump(resultados_finales, f, ensure_ascii=False, indent=4)

    print(f"\n¡Batería de Test completada! Se han generado {len(resultados_finales)} preguntas.")
    print(f"Guardado en: {ruta_salida}")

if __name__ == "__main__":
    procesar_test()

INICIANDO GENERACIÓN DE PREGUNTAS (TEST) CON GEMINI...
[1/78] Procesando bloque de: 1181_Guía hospitalaria de tera...
[2/78] Procesando bloque de: 1181_Guía hospitalaria de tera...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4515.82ms


[3/78] Procesando bloque de: 1181_Guía hospitalaria de tera...
[4/78] Procesando bloque de: 1181_Guía hospitalaria de tera...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2302.23ms


[5/78] Procesando bloque de: 1181_Guía hospitalaria de tera...
[6/78] Procesando bloque de: 1181_Guía hospitalaria de tera...


...

[75/78] Procesando bloque de: 16304_Protocolo para la admini...
[76/78] Procesando bloque de: 16304_Protocolo para la admini...
[77/78] Procesando bloque de: 11864_Procedimiento para el cu...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1517.64ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 785.18ms


    [Aviso] Cita un poco larga detectada (502 chars) en tipo 'Compleja'
[78/78] Procesando bloque de: 10604_Protocolo de uso de Nivo...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3439.10ms



¡Batería de Test completada! Se han generado 312 preguntas.
Guardado en: preguntas_evaluacion_test.json


In [11]:
import json
import re
import os
from thefuzz import fuzz

# ==========================================
# CONFIGURACIÓN DE ARCHIVOS
# ==========================================
ARCHIVO_PREGUNTAS = 'preguntas_evaluacion_test.json'
ARCHIVO_DATASET_TEST = 'dataset_test.json'
ARCHIVO_SALIDA = 'preguntas_evaluacion_test_final.json'

ESTRATEGIAS = {
    "A (Fixed)": 'chunks_estrategia_A_fixed.json',
    "B (Markdown)": 'chunks_estrategia_B_markdown.json',
    "C (Semántica)": 'chunks_estrategia_C_semantica.json',
    "D (Jerárquica)": 'chunks_estrategia_D_jerarquico.json'
}

UMBRAL_FUZZY = 90  # % de similitud mínima

def procesar_cita_multisalto(cita_llm):
    """Divide las citas si Gemini usó separadores (Ej: Concepto A | ... | Concepto B)"""
    if cita_llm in ["NO APLICA", "NULL", ""]:
        return []
    fragmentos = re.split(r'\.\.\.|\[\.\.\.\]', cita_llm)
    return [f.strip().lower() for f in fragmentos if len(f.strip()) > 5]

def cargar_json(ruta):
    """
    Carga un JSON desde disco, devolviendo una lista vacía si el archivo no
    existe.
    """
    if not os.path.exists(ruta):
        print(f"[ERROR] No se encuentra el archivo: {ruta}")
        return []
    with open(ruta, 'r', encoding='utf-8') as f:
        return json.load(f)

def auditar_dataset():
    """
    Realiza una doble auditoría del dataset de test:
    (1) Fidelidad: descarta las preguntas cuya cita_literal no se localiza
    de forma suficientemente fiel (fuzzy matching) en su macro-contexto de
    origen, o cuyos fragmentos superan los 500 caracteres.
    (2) Supervivencia al chunking: para cada una de las cuatro estrategias
    de la Fase 3b, comprueba si la cita fiel sigue siendo recuperable dentro
    de un único chunk o de un par de chunks adyacentes del documento, e
    informa la tasa de recuperabilidad máxima y las citas que el chunking
    ha destruido por completo.
    """
    print("="*70)
    print(" INICIANDO DOBLE AUDITORÍA DE DATASET RAG ".center(70))
    print("="*70)

    preguntas = cargar_json(ARCHIVO_PREGUNTAS)
    dataset_test = cargar_json(ARCHIVO_DATASET_TEST)
    datasets_chunks = {nombre: cargar_json(ruta) for nombre, ruta in ESTRATEGIAS.items()}

    if not preguntas or not dataset_test:
        return

    # Crear diccionario rápido para la Fidelidad: {macro_id: texto_original}
    diccionario_macro_contextos = {bloque["macro_id"]: bloque["text"].lower() for bloque in dataset_test}

    preguntas_evaluables = [p for p in preguntas if p.get("cita_literal", "") not in ["NO APLICA", "NULL", ""]]
    total_evaluables = len(preguntas_evaluables)

    preguntas_negativas = [p for p in preguntas if p.get("cita_literal", "") in ["NO APLICA", "NULL", ""]]

    # ==========================================
    # FASE 1: AUDITORÍA DE FIDELIDAD (LLM vs Texto Original)
    # ==========================================
    preguntas_fieles = []
    alucinaciones = 0

    for p in preguntas_evaluables:
        cita_original = p.get("cita_literal", "")
        macro_id = p.get("macro_id_origen", "")
        sub_citas = procesar_cita_multisalto(cita_original)
        tipo = p.get("tipo", "")  # Tipo de pregunta, usado en el filtro de longitud siguiente

        # ── FILTRO DE LONGITUD ──────────────────────────────────────────
        if tipo == "Compleja":
            if any(len(f) > 500 for f in sub_citas):
                print(f"  [DESCARTADA por longitud] Fragmento individual > 500 chars en Compleja")
                alucinaciones += 1
                continue
        else:
            if len(cita_original) > 500:
                print(f"  [DESCARTADA por longitud] Cita > 500 chars en {tipo}")
                alucinaciones += 1
                continue

        texto_base = diccionario_macro_contextos.get(macro_id, "")

        # ¿Están TODAS las sub-citas en el texto original que leyó Gemini?
        es_fiel = all(fuzz.partial_ratio(sub, texto_base) >= UMBRAL_FUZZY for sub in sub_citas)

        if es_fiel:
            # Guardamos las sub_citas procesadas para no repetir el cálculo en la Fase 2
            p["sub_citas_procesadas"] = sub_citas
            preguntas_fieles.append(p)
        else:
            alucinaciones += 1

    preguntas_finales = preguntas_fieles + preguntas_negativas

    with open(ARCHIVO_SALIDA, 'w', encoding='utf-8') as f:
        json.dump(preguntas_finales, f, ensure_ascii=False, indent=4)

    print(f"Limpieza completada.")
    print(f"   - Preguntas originales: {len(preguntas)}")
    print(f"   - Preguntas evaluables: {total_evaluables}")
    print(f"   - Alucinaciones eliminadas: {alucinaciones}")
    print(f"   - DATASET FINAL: {len(preguntas_finales)} preguntas. Guardado en '{ARCHIVO_SALIDA}'.")
    print("="*70)

    # ==========================================
    # FASE 2: AUDITORÍA DE SUPERVIVENCIA (Chunks vs Cita Fiel)
    # ==========================================
    resultados = {}

    for nombre_est, chunks_estrategia in datasets_chunks.items():
        if not chunks_estrategia: continue

        aciertos_individual = 0
        aciertos_pares = 0
        fallos_chunking = 0

        for p in preguntas_fieles:
            fuente_doc = p.get("source", "")
            sub_citas = p["sub_citas_procesadas"]

            # Filtramos los chunks de este documento usando el esquema exacto que pasaste
            # Filtramos los chunks de este documento
            textos_del_doc = []
            for c in chunks_estrategia:
                meta = c.get("metadata", {})
                if meta.get("source", "") == fuente_doc:
                    # REGLA DE ORO PARA LA JERÁRQUICA: Solo evaluamos los Hijos (los que tienen parent_id)
                    if nombre_est == "D (Jerárquica)":
                        if meta.get("doc_type") == "hijo":
                            textos_del_doc.append(c.get("text", ""))
                    else:
                        textos_del_doc.append(c.get("text", ""))

            # --- PRUEBA 2.1: SUPERVIVENCIA EN 1 CHUNK INDIVIDUAL ---
            sobrevive_individual = True
            for sub in sub_citas:
                encontrada_sub = any(fuzz.partial_ratio(sub, t.lower()) >= UMBRAL_FUZZY for t in textos_del_doc)
                if not encontrada_sub:
                    sobrevive_individual = False
                    break

            if sobrevive_individual:
                aciertos_individual += 1
                continue

            # --- PRUEBA 2.2: SUPERVIVENCIA EN PARES ADYACENTES ---
            sobrevive_pares = True
            for sub in sub_citas:
                encontrada_sub = False
                for i in range(len(textos_del_doc) - 1):
                    par_texto = (textos_del_doc[i] + " " + textos_del_doc[i+1]).lower()
                    if fuzz.partial_ratio(sub, par_texto) >= UMBRAL_FUZZY:
                        encontrada_sub = True
                        break
                if not encontrada_sub:
                    sobrevive_pares = False
                    break

            if sobrevive_pares:
                aciertos_pares += 1
            else:
                fallos_chunking += 1

        resultados[nombre_est] = {
            "individual": aciertos_individual,
            "pares": aciertos_pares,
            "destruidas": fallos_chunking
        }

    # ==========================================
    # IMPRIMIR INFORME FINAL
    # ==========================================
    total_validas = len(preguntas_fieles)

    for nombre, res in resultados.items():
        supervivencia_total = res['individual'] + res['pares']
        print(f"\n[ ESTRATEGIA: {nombre} ]")
        print(f" -> Supervivencia en 1 solo Chunk:      {res['individual']}/{total_validas} ({(res['individual']/max(1,total_validas))*100:.1f}%)")
        print(f" -> Supervivencia en Chunks Adyacentes: {res['pares']}/{total_validas} ({(res['pares']/max(1,total_validas))*100:.1f}%)")
        print(f" -> TASA DE RECUPERABILIDAD MÁXIMA:     {supervivencia_total}/{total_validas} ({(supervivencia_total/max(1,total_validas))*100:.1f}%)")
        print(f" -> Citas destruidas por la limpieza:   {res['destruidas']}")

if __name__ == "__main__":
    auditar_dataset()

               INICIANDO DOBLE AUDITORÍA DE DATASET RAG               
  [DESCARTADA por longitud] Cita > 500 chars en Caso
Limpieza completada.
   - Preguntas originales: 312
   - Preguntas evaluables: 234
   - Alucinaciones eliminadas: 7
   - DATASET FINAL: 305 preguntas. Guardado en 'preguntas_evaluacion_test_final.json'.

[ ESTRATEGIA: A (Fixed) ]
 -> Supervivencia en 1 solo Chunk:      226/227 (99.6%)
 -> Supervivencia en Chunks Adyacentes: 0/227 (0.0%)
 -> TASA DE RECUPERABILIDAD MÁXIMA:     226/227 (99.6%)
 -> Citas destruidas por la limpieza:   1

[ ESTRATEGIA: B (Markdown) ]
 -> Supervivencia en 1 solo Chunk:      221/227 (97.4%)
 -> Supervivencia en Chunks Adyacentes: 0/227 (0.0%)
 -> TASA DE RECUPERABILIDAD MÁXIMA:     221/227 (97.4%)
 -> Citas destruidas por la limpieza:   6

[ ESTRATEGIA: C (Semántica) ]
 -> Supervivencia en 1 solo Chunk:      214/227 (94.3%)
 -> Supervivencia en Chunks Adyacentes: 12/227 (5.3%)
 -> TASA DE RECUPERABILIDAD MÁXIMA:     226/227 (99.6%)
 -> C

In [10]:
import json
import statistics
import random
from collections import Counter

ARCHIVO_PREGUNTAS = 'preguntas_evaluacion_test_final.json'

def calcular_metricas(lista_textos, etiqueta):
    """Calcula e imprime las métricas de longitud de una lista de textos."""
    longitudes = [len(t) for t in lista_textos if t] # Evitamos textos vacíos por seguridad

    if not longitudes:
        print(f" -> {etiqueta}: No hay datos suficientes.")
        return

    print(f" -> {etiqueta}:")
    print(f"      Media:   {statistics.mean(longitudes):.0f} caracteres")
    print(f"      Mediana: {statistics.median(longitudes):.0f} caracteres")
    print(f"      Máxima:  {max(longitudes)} caracteres")
    print(f"      Mínima:  {min(longitudes)} caracteres")

def auditar_granularidad_preguntas():
    """
    Calcula y muestra estadísticas de granularidad del dataset de test
    final (Golden Dataset): distribución por tipo de pregunta, longitud de
    preguntas, citas y respuestas ideales, control de integridad de campos
    y una muestra aleatoria.
    """
    print("="*80)
    print(" INICIANDO ANÁLISIS DE GRANULARIDAD DEL GOLDEN DATASET ".center(80))
    print("="*80)

    try:
        with open(ARCHIVO_PREGUNTAS, 'r', encoding='utf-8') as f:
            preguntas = json.load(f)
    except FileNotFoundError:
        print(f"[X] Archivo no encontrado: {ARCHIVO_PREGUNTAS}")
        return

    if not preguntas:
        print("El dataset está vacío.")
        return

    # --- 1. MÉTRICAS DE DISTRIBUCIÓN ---
    print("\n[ 1. DISTRIBUCIÓN POR TIPO DE PREGUNTA ]")
    print("-" * 50)
    print(f" -> Total de preguntas validadas: {len(preguntas)}")

    tipos = [p.get("tipo", "Desconocido") for p in preguntas]
    conteo_tipos = Counter(tipos)

    for t, count in conteo_tipos.items():
        print(f"      - {t}: {count} preguntas ({(count/len(preguntas))*100:.1f}%)")

    # --- 2. MÉTRICAS DE LONGITUD LÉXICA ---
    print("\n[ 2. GRANULARIDAD LÉXICA (Caracteres) ]")
    print("-" * 50)

    # Extraemos las listas de textos
    textos_preguntas = [p.get("pregunta", "") for p in preguntas]
    textos_respuestas = [p.get("respuesta_ideal", "") for p in preguntas]

    # Para las citas, EXCLUIMOS las negativas ("NO APLICA", "NULL", "") para no arruinar la media
    textos_citas = [p.get("cita_literal", "") for p in preguntas if p.get("cita_literal", "") not in ["NO APLICA", "NULL", ""]]

    calcular_metricas(textos_preguntas, "Longitud de las PREGUNTAS (Prompt del usuario)")
    print("")
    calcular_metricas(textos_citas, "Longitud de las CITAS LITERALES (Ground Truth)")
    print("")
    calcular_metricas(textos_respuestas, "Longitud de las RESPUESTAS IDEALES (Gold Answer)")

    # --- 3. CONTROL DE CALIDAD Y ESTRUCTURA ---
    print("\n[ 3. CONTROL DE CALIDAD Y ESTRUCTURA ]")
    print("-" * 50)

    claves_base = ["tipo", "pregunta", "cita_literal", "respuesta_ideal", "macro_id_origen", "source"]
    muestra_p = preguntas[0]
    claves_ok = all(k in muestra_p for k in claves_base)
    print(f" -> Integridad de claves base: {'OK (Listas para Elasticsearch)' if claves_ok else 'ERROR: Faltan campos clave'}")

    # --- 4. MUESTRA VISUAL ---
    muestra = random.choice(preguntas)
    print("\n[ MUESTRA AL AZAR ]")
    print("-" * 50)
    print(f" -> Tipo:      {muestra.get('tipo')}")
    print(f" -> Documento: {muestra.get('source')}")
    print(f" -> Pregunta:  \"{muestra.get('pregunta')}\"")

    cita_corta = muestra.get('cita_literal', '')
    if len(cita_corta) > 100: cita_corta = cita_corta[:100] + "..."
    print(f" -> Cita:      \"{cita_corta}\"")
    print("="*80)

if __name__ == "__main__":
    auditar_granularidad_preguntas()

             INICIANDO ANÁLISIS DE GRANULARIDAD DEL GOLDEN DATASET              

[ 1. DISTRIBUCIÓN POR TIPO DE PREGUNTA ]
--------------------------------------------------
 -> Total de preguntas validadas: 305
      - Directa: 77 preguntas (25.2%)
      - Caso: 75 preguntas (24.6%)
      - Compleja: 75 preguntas (24.6%)
      - Negativa: 78 preguntas (25.6%)

[ 2. GRANULARIDAD LÉXICA (Caracteres) ]
--------------------------------------------------
 -> Longitud de las PREGUNTAS (Prompt del usuario):
      Media:   170 caracteres
      Mediana: 163 caracteres
      Máxima:  325 caracteres
      Mínima:  62 caracteres

 -> Longitud de las CITAS LITERALES (Ground Truth):
      Media:   264 caracteres
      Mediana: 254 caracteres
      Máxima:  701 caracteres
      Mínima:  46 caracteres

 -> Longitud de las RESPUESTAS IDEALES (Gold Answer):
      Media:   184 caracteres
      Mediana: 178 caracteres
      Máxima:  399 caracteres
      Mínima:  57 caracteres

[ 3. CONTROL DE CALIDAD Y E